In [4]:
import pandas as pd

file_path = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\flow_level\DrDoS_DNS_filtered.parquet"

df = pd.read_parquet(file_path, engine='pyarrow', columns=[' Timestamp'])

In [11]:
print(df[' Timestamp'])

0          2018-12-01 10:51:39.813448
1          2018-12-01 10:51:39.852499
2          2018-12-01 10:51:39.890213
3          2018-12-01 10:51:39.941151
4          2018-12-01 10:51:39.942030
                      ...            
5072524    2018-12-01 11:22:40.253588
5072525    2018-12-01 11:22:40.253659
5072526    2018-12-01 11:22:40.253852
5072527    2018-12-01 11:22:40.254534
5072528    2018-12-01 11:22:40.254719
Name:  Timestamp, Length: 5072529, dtype: object


In [25]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd

INPUT_FILE = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\flow_level\flow_level_original\TFTP_filtered.parquet"
OUTPUT_FILE = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\flow_level\5k_samples\TFTP_5k_samples.parquet"

SAMPLE_SIZE = 5000
CHUNK_SIZE = 30000

candidates = pd.DataFrame()

parquet_file = pq.ParquetFile(INPUT_FILE)

for batch in parquet_file.iter_batches(batch_size=CHUNK_SIZE):
    chunk = batch.to_pandas()

    chunk_sorted = chunk.sort_values(by=" Timestamp").head(SAMPLE_SIZE)

    candidates = pd.concat([candidates, chunk_sorted])

    candidates = candidates.sort_values(by=" Timestamp").head(SAMPLE_SIZE)

candidates.to_parquet(OUTPUT_FILE, index=False)
print(f"Earliest {SAMPLE_SIZE} rows by timestamp saved to {OUTPUT_FILE}")

Earliest 5000 rows by timestamp saved to C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\flow_level\5k_samples\TFTP_5k_samples.parquet


In [1]:
import os
import pandas as pd

INPUT_DIR = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\flow_level\5k_samples"

flow_file_map = {}  # Flow ID -> set of files it appears in

for file_name in os.listdir(INPUT_DIR):
    if not file_name.endswith(".parquet"):
        continue

    file_path = os.path.join(INPUT_DIR, file_name)
    df = pd.read_parquet(file_path, columns=["Flow ID"])

    for flow_id in df["Flow ID"]:
        if flow_id not in flow_file_map:
            flow_file_map[flow_id] = set()
        flow_file_map[flow_id].add(file_name)

# Collect Flow IDs that appear in more than one file
duplicate_flows = {fid: files for fid, files in flow_file_map.items() if len(files) > 1}

print(f"Total Flow IDs appearing in multiple files: {len(duplicate_flows)}\n")

# Example output: first 10 duplicates
for i, (fid, files) in enumerate(duplicate_flows.items()):
    print(f"{fid} -> {files}")
    if i >= 9:
        break

Total Flow IDs appearing in multiple files: 2222

192.168.50.6_23.194.142.213_56131_443_6 -> {'Benign_5k_samples.parquet', 'DrDoS_NTP_5k_samples.parquet'}
192.168.50.6_72.21.91.29_56123_80_6 -> {'Benign_5k_samples.parquet', 'DrDoS_NTP_5k_samples.parquet'}
192.168.50.6_8.43.72.98_56133_443_6 -> {'Benign_5k_samples.parquet', 'DrDoS_NTP_5k_samples.parquet'}
192.168.50.8_23.194.142.15_58323_443_6 -> {'Benign_5k_samples.parquet', 'DrDoS_NTP_5k_samples.parquet'}
192.168.50.7_4.2.2.4_55491_53_17 -> {'Benign_5k_samples.parquet', 'DrDoS_NTP_5k_samples.parquet'}
192.168.50.7_40.76.207.208_50474_443_6 -> {'Benign_5k_samples.parquet', 'DrDoS_NTP_5k_samples.parquet'}
192.168.50.7_4.2.2.4_54594_53_17 -> {'Benign_5k_samples.parquet', 'DrDoS_NTP_5k_samples.parquet'}
192.168.50.7_4.2.2.4_56216_53_17 -> {'Benign_5k_samples.parquet', 'DrDoS_NTP_5k_samples.parquet'}
192.168.50.7_23.32.248.65_50478_80_6 -> {'Benign_5k_samples.parquet', 'DrDoS_NTP_5k_samples.parquet'}
192.168.50.7_8.8.8.8_54594_53_17 -> {'B